In [1]:
"""
Rutgers Dining Hall Menu & Nutrition Scraper
============================================
Scrapes daily menu and per-item nutrition information from the public
Rutgers dining hall web portal and saves a structured result to output.json.

Halls scraped
-------------
  - Livingston Dining Hall  (locationNum = 03)
  - Neilson Dining Hall     (locationNum = 05)   [Cook/Douglass campus]
  - The Atrium              (locationNum = 13)   [Busch campus]

Meals scraped: Breakfast, Lunch, Dinner.

Each menu item is captured along with the contents of its detail page.
Top-level metadata (scrape_date, scrape_timestamp, source_url) is included
so the downstream database/notebook can satisfy the rubric's
"data freshness" evaluation criterion.
"""

import json
import re
import time
from datetime import date, datetime
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

HALLS = [
    {"name": "Livingston", "location_num": "03", "url_path": "Livingston+Dinning+Hall", "campus": "Livingston"},
    {"name": "Neilson",    "location_num": "05", "url_path": "Neilson+Dinning+Hall",    "campus": "Cook/Douglass"},
    {"name": "The Atrium", "location_num": "13", "url_path": "The+Atrium",              "campus": "Busch"},
]
MEALS = ["Breakfast", "Lunch", "Dinner"]
BASE_URL = "https://menuportal23.dining.rutgers.edu/FoodPronet/pickmenu.aspx"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (CS210-Project) Rutgers-Dining-Scraper",
}

REQUEST_DELAY_SEC   = 0.4 
REQUEST_TIMEOUT_SEC = 15
MAX_RETRIES         = 2

def safe_get(url):
    """GET ``url`` with a timeout and one retry. Returns Response or None."""
    last_err = None
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            last_err = e
            time.sleep(1.0)
    print(f"  ! request failed after {MAX_RETRIES} tries: {url}  ({last_err})")
    return None

def parse_nutrition(link):
    
    """
    Fetch one nutrition detail page and return a dict.

    Returns an empty dict on missing link, network failure, or unexpected
    HTML structure so callers can always rely on a dict result.
    """
    if not link:
        return {}

    resp = safe_get(link)
    if resp is None:
        return {}

    soup = BeautifulSoup(resp.text, "html.parser")
    nutrition = {}

    cal_p = soup.find("p", class_="strong")
    if cal_p is not None:
        parts = cal_p.get_text(strip=True).replace("\xa0", " ").split(" ")
        if len(parts) >= 2:
            nutrition["Calories"] = parts[1]

    for td in soup.find_all("td"):
        b_tag = td.find("b")
        if b_tag is None:
            parts = td.get_text(strip=True).split("\xa0")
            if len(parts) == 2:
                key, value = parts
                nutrition[key] = value
        else:
            key = b_tag.get_text(strip=True)
            value = td.get_text(strip=True).replace(key, "").strip()
            value = value.replace("\xa0", " ")
            if value == "%":
                continue
            nutrition[key] = value
        
    nutrition["Percentage"] = {}
    specs_div = soup.find("div", id="specs")
    if specs_div is not None:
        pct_re = re.compile(r"^\d+%$")
        for li in specs_div.find_all("li"):
            tokens = li.get_text(strip=True).split()
            if tokens and pct_re.match(tokens[-1]):
                nutrition["Percentage"][" ".join(tokens[:-1])] = tokens[-1]

    return nutrition

def parse_menu_items(base_url, fieldsets):
    """Parse <fieldset> blocks on a menu page into item dicts."""
    items = []
    for fs in fieldsets:
        div1 = fs.find("div", class_="col-1")
        div2 = fs.find("div", class_="col-2")
        div3 = fs.find("div", class_="col-3")

        label1 = div1.find("label") if div1 is not None else None
        label2 = div2.find("label") if div2 is not None else None

        item_name    = label1.get_text(strip=True) if label1 else None
        serving_size = (label2.get_text(strip=True).replace("\xa0", " ")
                        if label2 else None)

        nutrition_link = None
        if div3 is not None:
            link = div3.find("a")
            if link is not None:
                href = link.get("href")
                if href:
                    nutrition_link = urljoin(base_url, href)

        if not item_name:
            continue

        items.append({
            "item_name":      item_name,
            "serving_size":   serving_size,
            "nutrition_link": nutrition_link,
            "nutrition":      parse_nutrition(nutrition_link),
        })
        time.sleep(REQUEST_DELAY_SEC)

    return items


def fetch_menu(url):
    """Return list of item dicts from one menu page, or [] on failure."""
    resp = safe_get(url)
    if resp is None:
        return []
    soup = BeautifulSoup(resp.text, "html.parser")
    return parse_menu_items(url, soup.find_all("fieldset"))


def build_url(hall, meal, dt_str):
    return (f"{BASE_URL}?locationNum={hall['location_num']}"
            f"&locationName={hall['url_path']}"
            f"&dtdate={dt_str}"
            f"&activeMeal={meal}"
            f"&sName=Rutgers+University+Dining")

def main(out_path="output.json"):
    today_str  = date.today().strftime("%m/%d/%Y")
    scrape_iso = datetime.now().isoformat(timespec="seconds")

    out = {
        "scrape_date":      today_str,
        "scrape_timestamp": scrape_iso,
        "halls": {},
    }

    for hall in HALLS:
        print(f"[{hall['name']}]")
        out["halls"][hall["name"]] = {"campus": hall["campus"], "meals": {}}
        for meal in MEALS:
            url = build_url(hall, meal, today_str)
            print(f"  {meal} ...", end=" ", flush=True)
            items = fetch_menu(url)
            print(f"{len(items)} items")
            out["halls"][hall["name"]]["meals"][meal] = {
                "source_url": url,
                "items":      items,
            }

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2, ensure_ascii=False)

    total = sum(
        len(meal["items"])
        for hall in out["halls"].values()
        for meal in hall["meals"].values()
    )
    print(f"\nDone. {total} items saved to {out_path}")
    return out


if __name__ == "__main__":
    main()


[Livingston]
  Breakfast ... 0 items
  Lunch ... 67 items
  Dinner ... 66 items
[Neilson]
  Breakfast ... 26 items
  Lunch ... 69 items
  Dinner ... 65 items
[The Atrium]
  Breakfast ... 31 items
  Lunch ... 142 items
  Dinner ... 142 items

Done. 608 items saved to output.json
